In [2]:
import pandas as pd
import numpy as np
from pgmpy.models import BayesianModel
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination, ApproxInference, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BDeuScore, K2Score, BicScore
from pgmpy.metrics import structure_score
from pgmpy.utils import get_example_model
from pgmpy.estimators import ScoreCache
from pgmpy.inference.CausalInference import CausalInference
import networkx as nx
import itertools
import math
import networkx as nx
import matplotlib.pyplot as plt

In [3]:
from same_decision_probability_calculation import *
from minimum_information_loss_partition import *
from utils import *

from monte_carlo_sdp import *

In [4]:
from pgmpy.utils import get_example_model

# Loading Models

In [5]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from pgmpy.models import NaiveBayes
from pgmpy.estimators import MaximumLikelihoodEstimator

# ── VOTING ──────────────────────────────────────────────────────────────────
voting = fetch_ucirepo(id=105)
df_voting = pd.concat([voting.data.features, voting.data.targets], axis=1)
df_voting.columns = [c.strip() for c in df_voting.columns]

# Replace '?' missing values — Naive Bayes needs complete data
df_voting = df_voting.replace('?', pd.NA).dropna()

# All values must be strings/categories for pgmpy
df_voting = df_voting.astype(str)

target_voting = 'Class'   # 'democrat' / 'republican'

voting_model = NaiveBayes()
voting_model.fit(df_voting, target_voting,
                 estimator=MaximumLikelihoodEstimator)

# ── CHESS ────────────────────────────────────────────────────────────────────
chess = fetch_ucirepo(id=22)
df_chess = pd.concat([chess.data.features, chess.data.targets], axis=1)
df_chess = df_chess.astype(str)

target_chess = 'skach' 

chess_model = NaiveBayes()
chess_model.fit(df_chess, target_chess,
                estimator=MaximumLikelihoodEstimator)

In [6]:
# cast models to pgmpy BayesianNetwork for compatibility with our code
voting_model = BayesianNetwork(voting_model.edges())
chess_model = BayesianNetwork(chess_model.edges())

# fit
voting_model.fit(df_voting, estimator=MaximumLikelihoodEstimator)
chess_model.fit(df_chess, estimator=MaximumLikelihoodEstimator)

In [7]:
# find binary variables in chess df
binary_vars_chess = [col for col in df_chess.columns if df_chess[col].nunique() == 2]
print(f"Binary variables in Chess dataset: {binary_vars_chess}")

Binary variables in Chess dataset: ['bkblk', 'bknwy', 'bkon8', 'bkona', 'bkspr', 'bkxbq', 'bkxcr', 'bkxwp', 'blxwp', 'bxqsq', 'cntxt', 'dsopp', 'dwipd', 'katri', 'mulch', 'qxmsq', 'r2ar8', 'reskd', 'reskr', 'rimmx', 'rkxwp', 'rxmsq', 'simpl', 'skach', 'skewr', 'skrxp', 'spcop', 'stlmt', 'thrsk', 'wkcti', 'wkna8', 'wknck', 'wkovl', 'wkpos', 'wtoeg']


In [8]:
alarm_model = get_example_model('alarm')
child_model = get_example_model('child')
#asia_model = get_example_model('asia')
insurance_model = get_example_model('insurance')
hailfinder_model = get_example_model('hailfinder')
hepar_model = get_example_model('hepar2')
barley_model = get_example_model('barley')
win95pts_model = get_example_model('win95pts')
#mildew_model = get_example_model('mildew')
#water_model = get_example_model('water')
mildew_model = None
water_model = None

In [9]:
child_model.name = 'child'
insurance_model.name = 'insurance'
alarm_model.name = 'alarm'
hepar_model.name = 'hepar'
hailfinder_model.name = 'hailfinder'
win95pts_model.name = 'win95pts'
barley_model.name = 'barley'
voting_model.name = 'voting'
chess_model.name = 'chess'
#mildew_model.name = 'mildew'
#water_model.name = 'water'

In [10]:
# cardinality of variables
for node in child_model.nodes():
    print(f"Cardinality of variable '{node}': {child_model.get_cardinality(node)}")

Cardinality of variable 'BirthAsphyxia': 2
Cardinality of variable 'HypDistrib': 2
Cardinality of variable 'HypoxiaInO2': 3
Cardinality of variable 'CO2': 3
Cardinality of variable 'ChestXray': 5
Cardinality of variable 'Grunting': 2
Cardinality of variable 'LVHreport': 2
Cardinality of variable 'LowerBodyO2': 3
Cardinality of variable 'RUQO2': 3
Cardinality of variable 'CO2Report': 2
Cardinality of variable 'XrayReport': 5
Cardinality of variable 'Disease': 6
Cardinality of variable 'GruntingReport': 2
Cardinality of variable 'Age': 3
Cardinality of variable 'LVH': 2
Cardinality of variable 'DuctFlow': 3
Cardinality of variable 'CardiacMixing': 4
Cardinality of variable 'LungParench': 3
Cardinality of variable 'LungFlow': 3
Cardinality of variable 'Sick': 2


# Run Experiment

In [11]:
def get_target(model):
    targets = {
        'child': 'Sick',
        'alarm': 'HYPOVOLEMIA',
        'barley': 'pesticid',
        'insurance': 'Theft',
        'mildew': None, # no binary variables
        'water': None, # no binary variables
        'hailfinder': 'ScenRelAMCIN',
        'hepar': 'hepatomegaly',
        'win95pts': 'PrtMem',
        'voting': 'Class',
        'chess': 'skach'
    }
    # define the target manually when avaiable (from the respective paper) or randomly
    # conferir se vão ser esses mesmos!!

    return targets[model.name]

def get_h_ratio(model):
    ratios = {
        'child': 0.5,
        'alarm': 0.30,
        'hepar': 0.20,
        'barley': 0.20,
        'mildew': 0.30,
        'water': 0.30,
        'hailfinder': 0.30,
        'win95pts': 0.20,
        'insurance': 0.40,
        'voting': 0.5,
        'chess': 0.9 #era 0.86
    }
    return ratios[model.name]
    



In [12]:
len(chess_model.nodes())

36

In [13]:
models_to_run = [voting_model, chess_model, child_model, alarm_model, insurance_model, hailfinder_model, hepar_model, win95pts_model]

In [14]:
#models_to_run = [chess_model]

In [15]:
for model in models_to_run:
    print(f"Model '{model.name}' has {len(model.nodes())} variables.")

Model 'voting' has 17 variables.
Model 'chess' has 36 variables.
Model 'child' has 20 variables.
Model 'alarm' has 37 variables.
Model 'insurance' has 27 variables.
Model 'hailfinder' has 56 variables.
Model 'hepar' has 70 variables.
Model 'win95pts' has 76 variables.


In [16]:
len(models_to_run)

8

In [17]:
all_targets_are_binary = True
for bn in models_to_run:
    #print(f"\n=== BN: {bn.name} ===")
    target = get_target(bn)
    if target is None:
        #print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    if len(target_states) != 2:
        #print(f"--> Target '{target}' in {bn.name} is not binary (States: {target_states}), skipping.")
        all_targets_are_binary = False
        continue
    #print(f"Available states for target '{target}': {target_states}")
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    #print(f"Target Node: {target}, Target Value: {target_value}")

print(f"\nAll targets are binary: {all_targets_are_binary}")


All targets are binary: True


In [18]:
for bn in models_to_run:
    print(f"\n=== BN: {bn.name} ===")
    all_nodes = list(bn.nodes())
    
    target = get_target(bn)
    if target is None:
        print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    print(f"Target Node: {target}, Target Value: {target_value}")
    
    available_nodes = [n for n in all_nodes if n != target]
    print(f"H ratio: {get_h_ratio(bn)}")
    n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
    print(f"using {n_hidden} H variables")


=== BN: voting ===
Target Node: Class, Target Value: republican
H ratio: 0.5
using 8 H variables

=== BN: chess ===
Target Node: skach, Target Value: t
H ratio: 0.9
using 31 H variables

=== BN: child ===
Target Node: Sick, Target Value: no
H ratio: 0.5
using 9 H variables

=== BN: alarm ===
Target Node: HYPOVOLEMIA, Target Value: FALSE
H ratio: 0.3
using 10 H variables

=== BN: insurance ===
Target Node: Theft, Target Value: False
H ratio: 0.4
using 10 H variables

=== BN: hailfinder ===
Target Node: ScenRelAMCIN, Target Value: CThruK
H ratio: 0.3
using 16 H variables

=== BN: hepar ===
Target Node: hepatomegaly, Target Value: absent
H ratio: 0.2
using 13 H variables

=== BN: win95pts ===
Target Node: PrtMem, Target Value: Less_than_2Mb
H ratio: 0.2
using 15 H variables


In [19]:
import time
from xml.parsers.expat import model
import tracemalloc

models_to_run = [child_model]

def run_for_time(func, *args, **kwargs):
    """Runs natively at maximum speed to record pure execution time."""
    start_time = time.time()
    try:
        result = func(*args, **kwargs)
        return result, (time.time() - start_time), True
    except Exception as e:
        return None, np.nan, False # Failed

def run_for_memory(func, *args, **kwargs):
    """Runs with tracemalloc to record peak memory. Ignores execution time."""
    tracemalloc.start()
    try:
        func(*args, **kwargs)
    except Exception:
        pass # We just want to see how high memory got before it crashed/finished
        
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak_mem / (1024 * 1024) # Return MB

def select_random_target(bn):
    # list all binary variables in the BN
    binary_vars = []
    for node in bn.nodes():
        cpd = bn.get_cpds(node)
        if cpd is not None and len(cpd.state_names[node]) == 2:
            binary_vars.append(node)
    if not binary_vars:
        return None, None
    print(f"Binary variables in {bn.name}: {binary_vars}")
    selected_target = random.choice(binary_vars)
    return selected_target

def run_targeted_sdp_experiment(output_csv="targeted_sdp_benchmark.csv"):
    
    results = []
    raw_results = []
    #H_RATIO = 0.20
    DECISION_THRESHOLD = 0.5
    TARGET_BUCKETS = [0.8]
    MCMC_TRIALS = 3 
    
    for bn in models_to_run:
        n_nodes = bn.number_of_nodes()
        print(f"\n========================================")
        print(f"Processing BN: {bn.name}")
        
        all_nodes = list(bn.nodes())
        
        target = get_target(bn)
        #target = select_random_target(bn)
        if target is None:
            print(f"--> No binary target defined for {bn.name}, skipping.")
            continue
        target_states = bn.get_cpds(target).state_names[target]
        target_value = target_states[1] if len(target_states) > 1 else target_states[0]
        print(f"Target Node: {target}, Target Value: {target_value}")
        
        available_nodes = [n for n in all_nodes if n != target]
        print(f"H ratio: {get_h_ratio(bn)}")
        #n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
        
        n_hidden = 16
        
        print(f"using {n_hidden} H variables")
        n_evidence = len(available_nodes) - n_hidden
        print(f"and {n_evidence} evidence variables")
        #hidden_vars = random.sample(available_nodes, n_hidden)
        #evidence_vars = [n for n in available_nodes if n not in hidden_vars]
        
            
        harvested_data = find_exact_experimental_patients_random(bn, target, target_value, DECISION_THRESHOLD,
                                                          n_evidence, buckets=TARGET_BUCKETS)
        
        # Now process whatever it managed to find
        for target_sdp, result in harvested_data.items():
            if result is None:
                continue # We didn't find a patient for this specific bucket in this network
                
            patient, exact_sdp = result
            hidden_vars = [n for n in bn.nodes() if n not in patient and n != target]
            print(f"\n  -> Benchmarking found patient for bucket {target_sdp} (Exact: {exact_sdp:.4f})")
            
            # ========================================================
            # EXACT SDP EVALUATION
            # ========================================================
            partitions = get_partitions(bn, hidden_vars, target, patient)
            print(f"       -> Running Exact SDP...")
            
            # Pass 1: Time
            exact_sdp, exact_time, exact_success = run_for_time(
                fast_broadcast_sdp, bn, target, target_value, patient, DECISION_THRESHOLD, partitions
            )
            
            # Pass 2: Memory
            exact_mem_mb = run_for_memory(
                fast_broadcast_sdp, bn, target, target_value, patient, DECISION_THRESHOLD, partitions
            )
            
            if exact_success:
                print(f"          Time: {exact_time:.4f} sec | Peak Memory: {exact_mem_mb:.2f} MB")
            else:
                print(f"          [FAILED]: Crashed at {exact_mem_mb:.2f} MB")

            # ========================================================
            # MCMC EVALUATION
            # ========================================================
            mcmc_estimates = []
            mcmc_times = []
            
            print(f"       -> Running MCMC SDP (Trials: {MCMC_TRIALS})...")
            
            # Pass 1: Pure Time (across all trials)
            for trial in range(MCMC_TRIALS):
                est_sdp, t_time, _ = run_for_time(
                    fast_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
                    n_samples=1000, burn_in=2000, thinning=50
                )
                mcmc_estimates.append(est_sdp)
                mcmc_times.append(t_time)
                
            mcmc_mean = np.mean(mcmc_estimates)
            mcmc_avg_time = np.mean(mcmc_times)
            mcmc_variance = np.var(mcmc_estimates)

            # Pass 2: Peak Memory
            mcmc_mem_mb = run_for_memory(
                fast_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
                n_samples=100, burn_in=50, thinning=5
            )
            
            print(f"          Avg Time: {mcmc_avg_time:.4f} sec | Peak Memory: {mcmc_mem_mb:.2f} MB")
            
            absolute_error = abs(exact_sdp - mcmc_mean)

            # ========================================================
            # PARALLEL TEMPERING MCMC EVALUATION
            # ========================================================

            pt_mcmc_estimates = []
            pt_mcmc_times = []

            print(f"       -> Running Parallel Tempering MCMC SDP (Trials: {MCMC_TRIALS})...")              
            
            # Pass 1: Pure Time (across all trials)
            for trial in range(MCMC_TRIALS):
                est_sdp, t_time, _ = run_for_time(
                    pt_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
                    n_samples=1000, burn_in=2000, thinning=50, n_chains=4, max_temp=40.0
                )
                pt_mcmc_estimates.append(est_sdp)
                pt_mcmc_times.append(t_time)

            pt_mcmc_mean = np.mean(pt_mcmc_estimates)
            pt_mcmc_avg_time = np.mean(pt_mcmc_times)
            pt_mcmc_variance = np.var(pt_mcmc_estimates)

            # Pass 2: Peak Memory
            pt_mcmc_mem_mb = run_for_memory(
                pt_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
                n_samples=100, burn_in=50, thinning=5, n_chains=4, max_temp=10.0
            )

            print(f"          Avg Time: {pt_mcmc_avg_time:.4f} sec | Peak Memory: {pt_mcmc_mem_mb:.2f} MB")

            absolute_error_pt = abs(exact_sdp - pt_mcmc_mean)
            
        
            # Record everything to the dataset
            results.append({
                'Network': bn.name,
                'N_Nodes': n_nodes,
                'Target_Bucket': target_sdp,
                'Target_Node': target,
                'Target_Value': target_value,
                'Exact_SDP': exact_sdp,
                'Exact_Time_sec': exact_time,
                'MCMC_Mean_SDP': mcmc_mean,
                'MCMC_Variance': mcmc_variance,
                'MCMC_Avg_Time_sec': mcmc_avg_time,
                'Absolute_Error': absolute_error,
                'PT_MCMC_Mean_SDP': pt_mcmc_mean,
                'PT_MCMC_Variance': pt_mcmc_variance,
                'PT_MCMC_Avg_Time_sec': pt_mcmc_avg_time,
                'PT_Absolute_Error': absolute_error_pt
            })
            
            # Save progressively
            pd.DataFrame(results).to_csv(output_csv, index=False)
            pd.DataFrame(raw_results).to_csv("raw_" + output_csv, index=False)

    print(f"\nExperiment Complete! Results saved to {output_csv}")
    return pd.DataFrame(results)

In [20]:
run_targeted_sdp_experiment()


Processing BN: child
Target Node: Sick, Target Value: no
H ratio: 0.5
using 16 H variables
and 3 evidence variables

Hunting for patients... (Locking 3 variables as evidence)
Generating batch 1/2 of 8000 random realities...
--> Filled bucket 0.8 with Exact SDP: 0.7850
All buckets filled successfully!

  -> Benchmarking found patient for bucket 0.8 (Exact: 0.7850)
       -> Running Exact SDP...
          Time: 4.2039 sec | Peak Memory: 961.11 MB
       -> Running MCMC SDP (Trials: 3)...
          Avg Time: 1.7698 sec | Peak Memory: 0.27 MB
       -> Running Parallel Tempering MCMC SDP (Trials: 3)...
          Avg Time: 6.4178 sec | Peak Memory: 0.32 MB

Experiment Complete! Results saved to targeted_sdp_benchmark.csv


,Network,N_Nodes,Target_Bucket,Target_Node,Target_Value,Exact_SDP,Exact_Time_sec,MCMC_Mean_SDP,MCMC_Variance,MCMC_Avg_Time_sec,Absolute_Error,PT_MCMC_Mean_SDP,PT_MCMC_Variance,PT_MCMC_Avg_Time_sec,PT_Absolute_Error
0,child,20,0.8,Sick,no,0.785049,4.203906,0.801,0.000165,1.769768,0.015951,0.791667,0.000039,6.417813,0.006618
